# Mushuk va It klassifikatori — ML jarayonini amalda ko'rish

Bu notebook — darsimizda gaplashgan **Ma'lumot → Train → Test → Evaluate** jarayonining haqiqiy kod bilan bajarilishi.

Biz kompyuterga rasmga qarab **"bu mushukmi yoki itmi?"** ni aniqlashni o'rgatamiz. Har bir qadamda **nima qilinayotgani** va **nega aynan shunday qilinayotgani** tushuntiriladi.

**Ma'lumotlar haqida:** bu notebook hech qanday tashqi papka yoki o'zingiz tayyorlagan rasm talab qilmaydi — **1-qadamda ma'lumot avtomatik yuklab olinadi** (mashhur, ochiq "Cats vs Dogs" to'plamidan kichik bir qism). Faylni istalgan joyga qo'yib, hujayralarni tartib bilan ishga tushirsangiz bo'ldi.

## 1-qadam: Kerakli kutubxonalarni yuklash

Kod yozishdan oldin, bizga kerak bo'ladigan "vositalar"ni chaqirib olamiz:

- **TensorFlow / Keras** — modelni qurish va o'qitish uchun asosiy kutubxona
- **matplotlib** — rasmlarni va grafiklarni ko'rsatish (vizualizatsiya) uchun
- **numpy** — sonlar bilan ishlash uchun
- **scikit-learn** — natijalarni baholash (accuracy, confusion matrix) uchun

*Bu — xuddi oshxonaga kirishdan oldin kerakli asboblarni stolga qo'yib chiqishga o'xshaydi.*

In [1]:
import os
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow versiyasi:", tf.__version__)


TensorFlow versiyasi: 2.20.0


## 2-qadam: Ma'lumotlarni avtomatik yuklab olish (Data loading)

Bu — darsimizdagi **birinchi bosqich: Ma'lumot to'plash**.

Biz Google tomonidan taqdim etilgan, mashhur **"Cats vs Dogs"** to'plamining kichik versiyasini (`cats_and_dogs_filtered`) avtomatik yuklab olamiz. Bu — `tf.keras.utils.get_file()` funksiyasi orqali amalga oshadi: fayl bir marta yuklanadi, keyingi safar ishga tushirganingizda esa xotiradan (cache'dan) darhol olinadi.

Yuklab olingandan so'ng, biz sinf uchun **tezkor demo maqsadida** har klassdan (mushuk/it) kichik bir qismini (masalan 25 tadan) tanlab olamiz — shunda video yozish paytida train jarayoni bir necha soniyada tugaydi. Xohlasangiz, `N_PER_CLASS` sonini oshirib, ko'proq rasm bilan ham sinab ko'rishingiz mumkin (natija shunchalik yaxshi bo'ladi, lekin train biroz uzoqroq davom etadi).

In [2]:
# 1) Ochiq ma'lumotlar to'plamini yuklab olish (bir marta yuklanadi, keyin cache'dan olinadi)
DATASET_URL = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
zip_path = tf.keras.utils.get_file("cats_and_dogs_filtered.zip", origin=DATASET_URL, extract=True)
base_dir = os.path.join(os.path.dirname(zip_path), "cats_and_dogs_filtered")
train_source = os.path.join(base_dir, "train")

print("Ma'lumot muvaffaqiyatli yuklandi:", base_dir)
print("Mavjud klasslar:", os.listdir(train_source))


Exception: URL fetch failure on https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip: 403 -- Forbidden

In [ ]:
# 2) Tezkor demo uchun kichik subset yasash (har klassdan N_PER_CLASS ta rasm)
N_PER_CLASS = 25  # tezkor demo uchun; ko'proq vaqtingiz bo'lsa bu sonni oshiring (masalan 100)
SUBSET_DIR = "cats_vs_dogs_subset"

def make_subset():
    if os.path.exists(SUBSET_DIR):
        shutil.rmtree(SUBSET_DIR)
    random.seed(123)
    for cls in ["cats", "dogs"]:
        src_dir = os.path.join(train_source, cls)
        all_files = os.listdir(src_dir)
        chosen = random.sample(all_files, min(N_PER_CLASS, len(all_files)))
        dst_dir = os.path.join(SUBSET_DIR, cls)
        os.makedirs(dst_dir, exist_ok=True)
        for fname in chosen:
            shutil.copy(os.path.join(src_dir, fname), os.path.join(dst_dir, fname))
        print(f"'{cls}' klassidan {len(chosen)} ta rasm tanlandi.")

make_subset()


### Endi tanlangan subsetni Train/Test'ga bo'lamiz

Bu yerda darsimizdagi **80/20 train/test split** amalga oshadi — `validation_split=0.2` degani ma'lumotning 80%i train, 20%i test (validation) uchun ajratiladi.

**Nega rasm o'lchamini bir xillashtiramiz (`image_size=(128,128)`)?** Chunki model doim bir xil "shakldagi" kirish kutadi — xuddi barcha o'quvchilar bir xil o'lchamdagi varaqqa imtihon yozishi kabi.

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 8
SEED = 123

train_ds = tf.keras.utils.image_dataset_from_directory(
    SUBSET_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    SUBSET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Klasslar:", class_names)
print("Klasslar mosligi -> 0 =", class_names[0], "| 1 =", class_names[1])


## 3-qadam: Ma'lumotni ko'zdan kechirish (vizualizatsiya)

**Muhim odat:** kodni davom ettirishdan oldin, ma'lumotingizga albatta bir marta ko'z bilan qarab chiqing. Bu xatolarni (masalan, noto'g'ri papkaga tushib qolgan rasm) oldindan aniqlashga yordam beradi.

Pastda train to'plamidan bir nechta namunani, ularning haqiqiy yorlig'i bilan birga ko'ramiz.

In [ ]:
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(min(8, len(images))):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.suptitle("Train to'plamidan namunalar")
plt.tight_layout()
plt.show()


## 4-qadam: Ishlash tezligini optimallashtirish

Bu qadam modelning ishlashiga emas, balki **tezligiga** ta'sir qiladi — `cache()` ma'lumotni xotirada saqlaydi, `prefetch()` esa keyingi partiyani oldindan tayyorlab qo'yadi. Kichik dataset uchun katta farq bo'lmasa-da, bu — professional Keras loyihalarida standart amaliyot.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(50).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)


## 5-qadam: Data augmentation — nega kerak?

Bizda har klassdan atigi **~25 tadan rasm** bor (tezkor demo uchun ataylab kichraytirilgan) — bu haqiqiy loyihalar uchun juda kam. Darsimizda aytganimizdek, kam ma'lumot modelni "yodlab qolish"ga (overfitting) olib kelishi mumkin.

**Data augmentation** — mavjud rasmlarni aylantirib, kattalashtirib, oyna kabi aks ettirib, sun'iy ravishda "xilma-xillik" qo'shish usuli. Bu modelga *"mushuk har doim bir xil burchakda turmaydi"* ekanini o'rgatishga yordam beradi.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])


## 6-qadam: Modelni qurish — Transfer Learning

Bizda har klassdan atigi ~25 tadan rasm bor. Agar noldan katta neyron tarmoq qursak, u hech narsani o'rgana olmaydi — ma'lumot juda kam.

Shuning uchun **Transfer Learning** (ko'chirib o'rganish) usulidan foydalanamiz: **MobileNetV2** — millionlab rasmda (ImageNet to'plamida) allaqachon o'qitilgan tayyor modeldan foydalanamiz. Bu model shakllarni, chiziqlarni, teksturalarni tanishni allaqachon biladi.

*Bu — xuddi noldan chizishni o'rganish o'rniga, tajribali rassomdan asosiy texnikalarni o'rganib, keyin faqat o'zingizning mavzuingizga (mushuk/it) moslashtirishga o'xshaydi.*

`base_model.trainable = False` — bu qatorda biz MobileNetV2ning bilimini **"muzlatib qo'yamiz"** (o'zgartirmaymiz), faqat uning ustiga kichik, o'zimizning "mushuk vs it" qarorini beruvchi qismni qo'shamiz.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=["accuracy"],
)

model.summary()


## 7-qadam: Modelni o'qitish (Train)

Bu — darsimizdagi **ikkinchi bosqich: Train**.

- **epoch** — model butun train ma'lumotini nechchi marta "qayta ko'rib chiqishi". Har epoch'da model biroz yaxshilanadi.
- **validation_data** — har epoch oxirida modelni test to'plamida ham tekshirib turamiz, shunda train paytida ham "imtihon natijasi"ni kuzatib boramiz.

Dataset juda kichik bo'lgani uchun ko'p epoch shart emas — 10 ta epoch yetarli.

In [ ]:
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
)


## 8-qadam: Train jarayonini vizualizatsiya qilish

Endi modelning epoch-epoch qanday o'rganganini grafikda ko'ramiz — accuracy (aniqlik) oshib borishi va loss (xato) kamayib borishi kerak.

**Diqqat qiling:** agar train accuracy juda yuqori bo'lib, val (test) accuracy past yoki beqaror bo'lsa — bu **overfitting** belgisi bo'lishi mumkin (darsda gaplashganimizdek, kichik dataset bilan bu tabiiy holat).

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Train accuracy")
plt.plot(epochs_range, val_acc, label="Test accuracy")
plt.legend(loc="lower right")
plt.title("Aniqlik (Accuracy)")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Train loss")
plt.plot(epochs_range, val_loss, label="Test loss")
plt.legend(loc="upper right")
plt.title("Xato (Loss)")

plt.show()


## 9-qadam: Test to'plamida baholash (Evaluate)

Bu — darsimizdagi **uchinchi va to'rtinchi bosqich: Test va Evaluate**.

Model endi **oldin ko'rmagan** rasmlarda (test to'plamida) sinaladi. Natijada chiqadigan **accuracy** — bizning asosiy baholash mezonimiz.

In [ ]:
test_loss, test_accuracy = model.evaluate(val_ds)
print(f"\nTest (baholash) aniqligi: {test_accuracy * 100:.1f}%")
print(f"Test xatoligi (loss): {test_loss:.3f}")


### Batafsil baholash: Confusion Matrix va Classification Report

Faqat umumiy accuracy'ni bilish yetarli emas — qaysi klassda ko'proq xato qilinganini ham ko'rish foydali. Buning uchun **confusion matrix** (chalkashlik matritsasi) va **classification report** dan foydalanamiz.

In [ ]:
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    preds = (preds > 0.5).astype(int).flatten()
    y_true.extend(labels.numpy())
    y_pred.extend(preds)

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Model bashorati")
plt.ylabel("Haqiqiy yorliq")
plt.title("Confusion Matrix")
plt.show()


## 10-qadam: Bashoratlarni vizual ko'rish

Eng tushunarli qism — modelning har bir test rasmiga bergan bashoratini, haqiqiy javob bilan yonma-yon ko'ramiz. **Yashil ramka** — to'g'ri topilgan, **qizil ramka** — xato.

Bu — darsda sizning simulyatoringizda ko'rgan ✓/✗ belgilarining, endi haqiqiy rasmlar bilan ko'rinishi.

In [ ]:
plt.figure(figsize=(12, 8))
i = 0
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    for img, true_label, pred in zip(images, labels, preds):
        if i >= 8:
            break
        pred_label = 1 if pred[0] > 0.5 else 0
        confidence = pred[0] if pred_label == 1 else 1 - pred[0]
        is_correct = pred_label == true_label.numpy()

        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(img.numpy().astype("uint8"))
        plt.axis("off")
        color = "green" if is_correct else "red"
        for spine in ["top", "bottom", "left", "right"]:
            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_color(color)
            ax.spines[spine].set_linewidth(4)
        plt.title(f"Haqiqiy: {class_names[true_label]}\nBashorat: {class_names[pred_label]} ({confidence*100:.0f}%)", fontsize=9)
        i += 1
    if i >= 8:
        break

plt.tight_layout()
plt.show()


## Xulosa

Ushbu notebookda biz darsimizdagi to'rtta bosqichni **haqiqiy kod** bilan bajardik:

| Bosqich | Bu yerda qanday amalga oshirildi |
|---|---|
| **Ma'lumot** | Ochiq "Cats vs Dogs" to'plamidan avtomatik yuklab olindi, har klassdan `N_PER_CLASS` (standart: 25) ta rasm tanlandi |
| **Train** | Model 80% ma'lumot asosida o'qitildi, Transfer Learning (MobileNetV2) yordamida |
| **Test** | Model qolgan 20% — oldin ko'rmagan ma'lumotda sinaldi |
| **Evaluate** | Accuracy, confusion matrix va vizual natijalar orqali baholandi |

**Muhim eslatma (darsda aytish uchun):** biz ataylab kichik subset (`N_PER_CLASS=25`) ishlatdik — video/dars paytida tez ishlashi uchun. Real loyihalarda odatda minglab rasm ishlatiladi. Shuning uchun bu yerdagi natijalar 100% barqaror bo'lmasligi mumkin — va bu ham darsimizdagi muhim tushunchani tasdiqlaydi: **kam ma'lumot — ishonchsiz natija demakdir**. `N_PER_CLASS` sonini oshirsangiz (masalan 100–200 ga), natija sezilarli darajada barqarorlashadi.

**Keyingi qadam sifatida sinab ko'rish mumkin:**
- Har klass uchun ko'proq rasm qo'shish
- `base_model.trainable = True` qilib, MobileNetV2ni ham "fine-tune" qilish (ilg'or mavzu)
- Boshqa hayvon turlarini qo'shib, ko'p klassli klassifikatorga aylantirish
